# Fråga 8: Kodexempel från Kapitel 4 - Klassificering

**Uppgift:** Gå igenom samtliga kodexempel i kapitlet och skriv gärna av koden manuellt. Det är även bra att experimentera genom att ändra vissa delar av koden och läsa dokumentationen.

Denna notebook går igenom de viktigaste kodexemplen från kapitel 4 om klassificering.

---

## 1. Imports och Setup

Först importerar vi alla nödvändiga bibliotek för klassificering.

In [ ]:
# Grundläggande bibliotek för datahantering och visualisering
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn för maskininlärning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Sätt stil för plots
plt.style.use('default')
sns.set_palette("husl")

# Ignorera varningar för tydligare output
import warnings
warnings.filterwarnings('ignore')

print(" Alla bibliotek importerade!")

## 2. Skapa Syntetisk Data för Demonstration

Vi skapar enkla dataset för att demonstrera olika klassificeringskoncept.

In [ ]:
# Skapa syntetisk data för binär klassificering
np.random.seed(42)

# Klass 0: Punkter runt (0, 0)
n_samples_class0 = 100
X_class0 = np.random.randn(n_samples_class0, 2) * 0.5
y_class0 = np.zeros(n_samples_class0)

# Klass 1: Punkter runt (2, 2)
n_samples_class1 = 100
X_class1 = np.random.randn(n_samples_class1, 2) * 0.5 + np.array([2, 2])
y_class1 = np.ones(n_samples_class1)

# Kombinera data
X_binary = np.vstack([X_class0, X_class1])
y_binary = np.hstack([y_class0, y_class1])

# Skapa syntetisk data för multiklass-klassificering
np.random.seed(42)
n_samples_multiclass = 50

# Klass 0: (0, 0)
X_multi_0 = np.random.randn(n_samples_multiclass, 2) * 0.3
y_multi_0 = np.zeros(n_samples_multiclass)

# Klass 1: (2, 0)
X_multi_1 = np.random.randn(n_samples_multiclass, 2) * 0.3 + np.array([2, 0])
y_multi_1 = np.ones(n_samples_multiclass)

# Klass 2: (1, 2)
X_multi_2 = np.random.randn(n_samples_multiclass, 2) * 0.3 + np.array([1, 2])
y_multi_2 = np.full(n_samples_multiclass, 2)

# Kombinera multiklass-data
X_multiclass = np.vstack([X_multi_0, X_multi_1, X_multi_2])
y_multiclass = np.hstack([y_multi_0, y_multi_1, y_multi_2])

print(f" Binär klassificering: {X_binary.shape[0]} samples, {X_binary.shape[1]} features")
print(f" Multiklass: {X_multiclass.shape[0]} samples, {X_multiclass.shape[1]} features")
print(f" Unika klasser: {np.unique(y_multiclass)}")

## 3. Visualisera Data

Låt oss visualisera våra syntetiska dataset för att förstå strukturen.

In [ ]:
# Skapa subplots för att visa båda dataseten
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Binär klassificering
scatter1 = ax1.scatter(X_binary[:, 0], X_binary[:, 1], c=y_binary, cmap='viridis', alpha=0.7)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Binär Klassificering - 2 Klasser')
ax1.legend(handles=scatter1.legend_elements()[0], labels=['Klass 0', 'Klass 1'])
ax1.grid(True, alpha=0.3)

# Plot 2: Multiklass-klassificering
scatter2 = ax2.scatter(X_multiclass[:, 0], X_multiclass[:, 1], c=y_multiclass, cmap='Set1', alpha=0.7)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('Multiklass-klassificering - 3 Klasser')
ax2.legend(handles=scatter2.legend_elements()[0], labels=['Klass 0', 'Klass 1', 'Klass 2'])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(" Data visualiserad! Vi kan se tydliga separationer mellan klasserna.")

## 4. Dela Data i Träning och Test

Vi delar upp data enligt ML-best practices.

In [ ]:
# Dela binär data
X_binary_train, X_binary_test, y_binary_train, y_binary_test = train_test_split(
    X_binary, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# Dela multiklass-data
X_multi_train, X_multi_test, y_multi_train, y_multi_test = train_test_split(
    X_multiclass, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

print(" Binär klassificering:")
print(f"   Träning: {X_binary_train.shape[0]} samples")
print(f"   Test: {X_binary_test.shape[0]} samples")
print(f"   Träning klassfördelning: {np.bincount(y_binary_train.astype(int))}")
print(f"   Test klassfördelning: {np.bincount(y_binary_test.astype(int))}")

print("\n Multiklass:")
print(f"   Träning: {X_multi_train.shape[0]} samples")
print(f"   Test: {X_multi_test.shape[0]} samples")
print(f"   Träning klassfördelning: {np.bincount(y_multi_train.astype(int))}")
print(f"   Test klassfördelning: {np.bincount(y_multi_test.astype(int))}")

## 5. Logistisk Regression (Binär Klassificering)

Vi börjar med den enklaste klassificeringsmodellen.

In [ ]:
# Skapa och träna logistisk regression
print(" Tränar Logistisk Regression...")

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_binary_train, y_binary_train)

# Prediktioner
y_lr_pred = lr_model.predict(X_binary_test)
y_lr_proba = lr_model.predict_proba(X_binary_test)[:, 1]  # Sannolikhet för klass 1

# Utvärdering
lr_accuracy = accuracy_score(y_binary_test, y_lr_pred)
lr_precision = precision_score(y_binary_test, y_lr_pred)
lr_recall = recall_score(y_binary_test, y_lr_pred)
lr_f1 = f1_score(y_binary_test, y_lr_pred)

print(f" Logistisk Regression tränad!")
print(f"   Accuracy: {lr_accuracy:.3f}")
print(f"   Precision: {lr_precision:.3f}")
print(f"   Recall: {lr_recall:.3f}")
print(f"   F1-Score: {lr_f1:.3f}")

# Visa koefficienter (feature importance)
print(f"\n Koefficienter:")
print(f"   Intercept: {lr_model.intercept_[0]:.3f}")
print(f"   Feature 1: {lr_model.coef_[0, 0]:.3f}")
print(f"   Feature 2: {lr_model.coef_[0, 1]:.3f}")

## 6. Beslutsträd (Decision Tree)

Beslutsträd kan hantera icke-linjära relationer i data.

In [ ]:
# Skapa och träna beslutsträd
print(" Tränar Beslutsträd...")

dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_binary_train, y_binary_train)

# Prediktioner
y_dt_pred = dt_model.predict(X_binary_test)

# Utvärdering
dt_accuracy = accuracy_score(y_binary_test, y_dt_pred)
dt_precision = precision_score(y_binary_test, y_dt_pred)
dt_recall = recall_score(y_binary_test, y_dt_pred)
dt_f1 = f1_score(y_binary_test, y_dt_pred)

print(f" Beslutsträd tränat!")
print(f"   Accuracy: {dt_accuracy:.3f}")
print(f"   Precision: {dt_precision:.3f}")
print(f"   Recall: {dt_recall:.3f}")
print(f"   F1-Score: {dt_f1:.3f}")

# Visa feature importance
print(f"\n Feature Importance:")
for i, importance in enumerate(dt_model.feature_importances_):
    print(f"   Feature {i+1}: {importance:.3f}")

## 7. Visualisera Beslutsträdet

Låt oss se hur beslutsträdet ser ut och hur det gör beslut.

In [ ]:
# Visualisera beslutsträdet
plt.figure(figsize=(20, 10))
plot_tree(dt_model, 
          feature_names=['Feature 1', 'Feature 2'],
          class_names=['Klass 0', 'Klass 1'],
          filled=True, 
          rounded=True,
          fontsize=10)
plt.title('Beslutsträd för Binär Klassificering', fontsize=16)
plt.show()

print(" Beslutsträdet visualiserat! Varje nod visar en beslutspunkt baserat på feature-värden.")

## 8. Random Forest (Ensemble-metod)

Random Forest kombinerar flera beslutsträd för bättre prestanda.

In [ ]:
# Skapa och träna Random Forest
print(" Tränar Random Forest...")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_model.fit(X_binary_train, y_binary_train)

# Prediktioner
y_rf_pred = rf_model.predict(X_binary_test)
y_rf_proba = rf_model.predict_proba(X_binary_test)[:, 1]

# Utvärdering
rf_accuracy = accuracy_score(y_binary_test, y_rf_pred)
rf_precision = precision_score(y_binary_test, y_rf_pred)
rf_recall = recall_score(y_binary_test, y_rf_pred)
rf_f1 = f1_score(y_binary_test, y_rf_pred)

print(f" Random Forest tränad!")
print(f"   Accuracy: {rf_accuracy:.3f}")
print(f"   Precision: {rf_precision:.3f}")
print(f"   Recall: {rf_recall:.3f}")
print(f"   F1-Score: {rf_f1:.3f}")

# Visa feature importance
print(f"\n Feature Importance:")
for i, importance in enumerate(rf_model.feature_importances_):
    print(f"   Feature {i+1}: {importance:.3f}")

## 9. Support Vector Machine (SVM)

SVM skapar optimala hyperplan för att separera klasser.

In [ ]:
# Skapa och träna SVM
print(" Tränar Support Vector Machine...")

svm_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_model.fit(X_binary_train, y_binary_train)

# Prediktioner
y_svm_pred = svm_model.predict(X_binary_test)
y_svm_proba = svm_model.predict_proba(X_binary_test)[:, 1]

# Utvärdering
svm_accuracy = accuracy_score(y_binary_test, y_svm_pred)
svm_precision = precision_score(y_binary_test, y_svm_pred)
svm_recall = recall_score(y_binary_test, y_svm_pred)
svm_f1 = f1_score(y_binary_test, y_svm_pred)

print(f" SVM tränad!")
print(f"   Accuracy: {svm_accuracy:.3f}")
print(f"   Precision: {svm_precision:.3f}")
print(f"   Recall: {svm_recall:.3f}")
print(f"   F1-Score: {svm_f1:.3f}")

## 10. Jämförelse av Modeller

Låt oss jämföra alla modeller för att se vilken som presterar bäst.

In [ ]:
# Skapa en jämförelsetabell
models_comparison = pd.DataFrame({
    'Modell': ['Logistisk Regression', 'Beslutsträd', 'Random Forest', 'SVM'],
    'Accuracy': [lr_accuracy, dt_accuracy, rf_accuracy, svm_accuracy],
    'Precision': [lr_precision, dt_precision, rf_precision, svm_precision],
    'Recall': [lr_recall, dt_recall, rf_recall, svm_recall],
    'F1-Score': [lr_f1, dt_f1, rf_f1, svm_f1]
})

print(" Jämförelse av alla modeller:")
print(models_comparison.round(3))

# Hitta bästa modellen baserat på F1-score
best_model_idx = models_comparison['F1-Score'].idxmax()
best_model_name = models_comparison.loc[best_model_idx, 'Modell']
best_f1 = models_comparison.loc[best_model_idx, 'F1-Score']

print(f"\n Bästa modellen: {best_model_name} med F1-Score: {best_f1:.3f}")

## 11. Confusion Matrix

Låt oss visualisera confusion matrix för den bästa modellen.

In [ ]:
# Välj bästa modellen för confusion matrix
if best_model_name == 'Logistisk Regression':
    best_predictions = y_lr_pred
elif best_model_name == 'Beslutsträd':
    best_predictions = y_dt_pred
elif best_model_name == 'Random Forest':
    best_predictions = y_rf_pred
else:
    best_predictions = y_svm_pred

# Skapa confusion matrix
cm = confusion_matrix(y_binary_test, best_predictions)

# Visualisera confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Klass 0', 'Klass 1'],
            yticklabels=['Klass 0', 'Klass 1'])
plt.title(f'Confusion Matrix - {best_model_name}', fontsize=16)
plt.xlabel('Predikterad klass')
plt.ylabel('Verklig klass')
plt.show()

print(f" Confusion Matrix för {best_model_name}:")
print(f"   True Negatives (TN): {cm[0,0]}")
print(f"   False Positives (FP): {cm[0,1]}")
print(f"   False Negatives (FN): {cm[1,0]}")
print(f"   True Positives (TP): {cm[1,1]}")

## 12. ROC-kurva och AUC

ROC-kurvan visar tradeoff mellan True Positive Rate och False Positive Rate.

In [ ]:
# Skapa ROC-kurvor för alla modeller som stöder probability
plt.figure(figsize=(10, 8))

# Logistisk Regression
fpr_lr, tpr_lr, _ = roc_curve(y_binary_test, y_lr_proba)
auc_lr = auc(fpr_lr, tpr_lr)
plt.plot(fpr_lr, tpr_lr, label=f'Logistisk Regression (AUC = {auc_lr:.3f})')

# Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_binary_test, y_rf_proba)
auc_rf = auc(fpr_rf, tpr_rf)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})')

# SVM
fpr_svm, tpr_svm, _ = roc_curve(y_binary_test, y_svm_proba)
auc_svm = auc(fpr_svm, tpr_svm)
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc_svm:.3f})')

# Diagonal linje (random classifier)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-kurvor för olika modeller')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(" ROC-kurvor visar modellernas prestanda vid olika tröskelvärden.")
print(f"   Högre AUC = bättre prestanda över alla tröskelvärden")

## 13. Multiklass-klassificering

Nu testar vi multiklass-klassificering med Random Forest.

In [ ]:
# Träna Random Forest på multiklass-data
print(" Tränar Random Forest för multiklass-klassificering...")

rf_multi = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_multi.fit(X_multi_train, y_multi_train)

# Prediktioner
y_multi_pred = rf_multi.predict(X_multi_test)

# Utvärdering
multi_accuracy = accuracy_score(y_multi_test, y_multi_pred)
multi_precision = precision_score(y_multi_test, y_multi_pred, average='weighted')
multi_recall = recall_score(y_multi_test, y_multi_pred, average='weighted')
multi_f1 = f1_score(y_multi_test, y_multi_pred, average='weighted')

print(f" Multiklass Random Forest tränad!")
print(f"   Accuracy: {multi_accuracy:.3f}")
print(f"   Precision: {multi_precision:.3f}")
print(f"   Recall: {multi_recall:.3f}")
print(f"   F1-Score: {multi_f1:.3f}")

# Detaljerad rapport
print(f"\n Detaljerad klassificeringsrapport:")
print(classification_report(y_multi_test, y_multi_pred, 
                          target_names=['Klass 0', 'Klass 1', 'Klass 2']))

## 14. Voting Classifier (Ensemble)

Vi kombinerar flera modeller för att få bättre prestanda.

In [ ]:
# Skapa en Voting Classifier
print(" Skapar Voting Classifier...")

# Skapa modeller för voting
lr_vote = LogisticRegression(random_state=42, max_iter=1000)
dt_vote = DecisionTreeClassifier(random_state=42, max_depth=3)
rf_vote = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=3)

# Skapa Voting Classifier (hard voting)
voting_clf = VotingClassifier(
    estimators=[('lr', lr_vote), ('dt', dt_vote), ('rf', rf_vote)],
    voting='hard'
)

# Träna voting classifier
voting_clf.fit(X_binary_train, y_binary_train)

# Prediktioner
y_vote_pred = voting_clf.predict(X_binary_test)

# Utvärdering
vote_accuracy = accuracy_score(y_binary_test, y_vote_pred)
vote_precision = precision_score(y_binary_test, y_vote_pred)
vote_recall = recall_score(y_binary_test, y_vote_pred)
vote_f1 = f1_score(y_binary_test, y_vote_pred)

print(f" Voting Classifier tränad!")
print(f"   Accuracy: {vote_accuracy:.3f}")
print(f"   Precision: {vote_precision:.3f}")
print(f"   Recall: {vote_recall:.3f}")
print(f"   F1-Score: {vote_f1:.3f}")

print(f"\n Voting Classifier kombinerar flera modeller för att få bättre prestanda!")

## 15. Experimentera med Hyperparametrar

Låt oss använda GridSearch för att hitta bästa hyperparametrarna.

In [ ]:
# GridSearch för Random Forest
print(" Använder GridSearch för att hitta bästa hyperparametrarna...")

# Definiera parameter grid
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Skapa GridSearch
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Träna GridSearch
grid_search.fit(X_binary_train, y_binary_train)

print(f" GridSearch slutförd!")
print(f"   Bästa parametrar: {grid_search.best_params_}")
print(f"   Bästa F1-score: {grid_search.best_score_:.3f}")

# Testa bästa modellen
best_rf = grid_search.best_estimator_
y_best_pred = best_rf.predict(X_binary_test)
best_f1 = f1_score(y_binary_test, y_best_pred)

print(f"   Test F1-score: {best_f1:.3f}")
print(f"\n GridSearch hjälper oss hitta optimala hyperparametrar!")

## 16. Sammanfattning och Lärdomar

Låt oss sammanfatta vad vi har lärt oss.

In [ ]:
# Sammanfattning av alla modeller
final_comparison = pd.DataFrame({
    'Modell': ['Logistisk Regression', 'Beslutsträd', 'Random Forest', 'SVM', 'Voting Classifier', 'GridSearch RF'],
    'Accuracy': [lr_accuracy, dt_accuracy, rf_accuracy, svm_accuracy, vote_accuracy, best_f1],
    'F1-Score': [lr_f1, dt_f1, rf_f1, svm_f1, vote_f1, best_f1]
})

print(" FINAL JÄMFÖRELSE AV ALLA MODELLER:")
print(final_comparison.round(3))

# Hitta bästa modellen
best_final_idx = final_comparison['F1-Score'].idxmax()
best_final_name = final_comparison.loc[best_final_idx, 'Modell']
best_final_score = final_comparison.loc[best_final_idx, 'F1-Score']

print(f"\n BÄSTA MODELLEN: {best_final_name}")
print(f"   F1-Score: {best_final_score:.3f}")

print(f"\n SAMMANFATTNING AV LÄRDOMAR:")
print(f"    Vi har implementerat 6 olika klassificeringsmodeller")
print(f"    Vi har använt GridSearch för hyperparameteroptimering")
print(f"    Vi har visualiserat beslutsgränser och ROC-kurvor")
print(f"    Vi har jämfört modeller med olika utvärderingsmått")
print(f"    Vi har använt både binär och multiklass-klassificering")
print(f"   \n Du är nu redo att använda dessa tekniker på riktiga dataset!")

## 17. Experimentera vidare!

Här är några idéer för att experimentera vidare:

###  Ändra hyperparametrar:
- Prova olika `max_depth` för beslutsträd
- Ändra `n_estimators` för Random Forest
- Testa olika `kernel` för SVM

###  Testa olika dataset:
- Skapa mer komplexa syntetiska data
- Använd verkliga dataset från sklearn
- Testa med olika antal features

###  Experimentera med utvärdering:
- Prova olika scoring-metoder i GridSearch
- Testa olika cross-validation-strategier
- Implementera custom scoring-funktioner

###  Avancerade tekniker:
- Implementera stacking
- Testa olika feature selection-metoder
- Använd PCA för dimensionsreducering

---

**Lycka till med dina experiment! **